# Bolt's Night Shift, Ep. 5 "Blind Corner": final render on Kaggle (free GPUs)

Renders all 1440 frames at 1920x1080 (Cycles, 64 samples, denoised) on Kaggle's free **2x T4** GPUs,
then adds captions and HUD, makes the 16:9 and 9:16 MP4s, the SRT and the thumbnails.
Full steps: `docs/RENDER_ON_KAGGLE.md` in the project.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**.
Then **Save Version > Save & Run All (Commit)**; it runs in the background (up to 12 h) and you can close the browser.

**Resuming:** if a run stops before all frames are done, open this notebook again, **Add Input > Your Work >**
this notebook's latest version, and **Save & Run All** again. Finished frames are reused, only the rest is rendered.

Results: **Output** tab of the version > `final/` (MP4s, SRT, thumbnails).

In [ ]:
# ---- settings -------------------------------------------------------------------------------
REPO = "https://github.com/Yashism/Blender-CC.git"
BRANCH = "claude/trusting-lovelace-345i7f"
SAMPLES = 64           # final quality; 32 renders ~2x faster with a little more denoiser softness
MAX_HOURS = 10.5       # Kaggle stops a session at 12 h: stop rendering in time to finish and save
CALIBRATE = False      # True: only time 6 frames on one GPU and print the ETA (about 10 minutes)
BLENDER_URL = "https://download.blender.org/release/Blender4.5/blender-4.5.14-linux-x64.tar.xz"

In [ ]:
# ---- setup: GPUs, Blender 4.5, project -----------------------------------------------------------
import glob, os, shutil, subprocess, time
WORK = "/kaggle/working"               # saved as the notebook output (20 GB max)
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
PROJ = f"{TMP}/blender-cc"
FRAMES = f"{WORK}/frames"
FINAL = f"{WORK}/final"
LOGS = f"{WORK}/logs"
for d in (FRAMES, FINAL, LOGS):
    os.makedirs(d, exist_ok=True)

gpus = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("\n".join(gpus) or "NO GPU: set Accelerator to GPU T4 x2 in the right-hand panel")
assert gpus, "no GPU"

BL = f"{TMP}/blender/blender"
if not os.path.exists(BL):
    subprocess.run(f"wget -q -O {TMP}/bl.tar.xz {BLENDER_URL} && mkdir -p {TMP}/blender && "
                   f"tar -xf {TMP}/bl.tar.xz -C {TMP}/blender --strip-components=1 && rm {TMP}/bl.tar.xz",
                   shell=True, check=True)
# libs Blender needs headless on the Kaggle image (usually already there)
subprocess.run("apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null 2>&1", shell=True)
print(subprocess.run([BL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

token = ""
try:                                    # only needed if the GitHub repo is private: Add-ons > Secrets > GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
if not os.path.isdir(PROJ):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, PROJ], check=True)
print(subprocess.run(["git", "-C", PROJ, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

# resume: reuse frames from an earlier version of this notebook attached as input
old = sorted(glob.glob("/kaggle/input/**/frames/f_*.png", recursive=True))
n = 0
for p in old:
    dst = os.path.join(FRAMES, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); n += 1
print(f"frames reused from earlier runs: {n}; frames present: {len(glob.glob(FRAMES + '/f_*.png'))}/1440")

In [ ]:
# ---- render: one Blender per GPU, each takes every n-th frame -----------------------------------
SCRIPT = f"{PROJ}/scripts/ep05/render_final.py"
base = [BL, "-b", "--factory-startup", "--python-exit-code", "1", "-P", SCRIPT, "--",
        "--device", "gpu", "--samples", str(SAMPLES), "--out", FRAMES]
if CALIBRATE:
    subprocess.run(base + ["--calibrate", "6"], env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    print(open(f"{PROJ}/renders/final/calibration/report.txt").read())
else:
    ng = len(gpus)
    procs = []
    for i in range(ng):
        lf = open(f"{LOGS}/gpu{i}.log", "a")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        procs.append((subprocess.Popen(base + ["--max-hours", str(MAX_HOURS), "--slice", f"{i}/{ng}"],
                                       stdout=lf, stderr=subprocess.STDOUT, env=env), i))
        time.sleep(90)                   # stagger the two scene builds (CPU and RAM)
    t0 = time.time()
    while any(p.poll() is None for p, _ in procs):
        time.sleep(300)
        done = len(glob.glob(FRAMES + "/f_*.png"))
        last = []
        for _, i in procs:
            lines = [l for l in open(f"{LOGS}/gpu{i}.log", errors="ignore") if l.startswith("[final]")]
            last.append(f"gpu{i}: " + (lines[-1].strip() if lines else "starting"))
        print(f"{(time.time() - t0) / 3600:.2f} h | {done}/1440 frames\n  " + "\n  ".join(last), flush=True)
    print("exit codes:", [p.returncode for p, _ in procs])
    print("frames present:", len(glob.glob(FRAMES + "/f_*.png")), "/ 1440")

In [ ]:
# ---- finish: captions + HUD, 16:9 + 9:16 MP4s, SRT, thumbnails (only when every frame exists) ----
done = len(glob.glob(FRAMES + "/f_*.png"))
if CALIBRATE:
    print("calibration only")
elif done < 1440:
    print(f"{done}/1440 frames. Run again (Add Input > this notebook's output, then Save & Run All) to render the rest.")
else:
    tmp_out = f"{TMP}/final"             # the captioned intermediates (~10 GB) stay off /kaggle/working
    rc = subprocess.run([BL, "-b", "--factory-startup", "--python-exit-code", "1", "-P",
                         f"{PROJ}/scripts/ep05/finish_final.py", "--",
                         "--src", FRAMES + "/f_%04d.png", "--out-dir", tmp_out]).returncode
    for p in glob.glob(tmp_out + "/*.mp4") + glob.glob(tmp_out + "/*.srt"):
        shutil.copy(p, FINAL)
    if os.path.isdir(tmp_out + "/thumbnails"):
        shutil.copytree(tmp_out + "/thumbnails", FINAL + "/thumbnails", dirs_exist_ok=True)
    print("finish exit code", rc)
    for p in sorted(glob.glob(FINAL + "/**", recursive=True)):
        if os.path.isfile(p):
            print(f"  {os.path.getsize(p) / 1e6:8.1f} MB  {p}")